# Análisis Exploratorio de Datos (EDA)

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Análisis general del dataset

## 1.1. Carga de datos

In [ ]:
df_train = pd.read_csv('../data/time-based-splits/train_before_eda.csv')
df_train.head()
df_train.tail()

## 1.2. Tipos de datos estructurados

### Clasificación de Variables
  
  | Variable | Significado | Tipo analítico |
  |---|---|---|
  | `hotel` | Tipo de hotel (City Hotel o Resort Hotel) | Categórica **nominal** (dicotómica)|
  | `is_canceled` | **Objetivo**: 1 = cancelada, 0 = no cancelada | Categórica **binaria** |
  | `lead_time` | Días transcurridos entre la reserva y la llegada | Numérica **discreta** |
  | `arrival_date_year` | Año de la fecha de llegada | Numérica **discreta** / Categórica **ordinal** |
  | `arrival_date_month` | Mes de la fecha de llegada | Categórica **ordinal** |
  | `arrival_date_week_number` | Número de semana del año de llegada | Numérica **discreta** |
  | `arrival_date_day_of_month` | Día del mes de la fecha de llegada | Numérica **discreta** |
  | `stays_in_weekend_nights` | Número de noches de fin de semana (sábado y/o domingo) que el cliente reservó para pernoctar. | Numérica **discreta** (conteo) |
  | `stays_in_week_nights` | Número de noches de semana (lunes a viernes) reservadas.| Numérica **discreta** (conteo) |
  | `adults` | Número de adultos | Numérica **discreta** (conteo) |
  | `children` | Número de niños | Numérica **discreta** (conteo) |
  | `babies` | Número de bebés | Numérica **discreta** (conteo) |
  | `meal` | Tipo de comida reservada  [**SC / Undefined**: Self Catering (sin comida incluida), **BB**: Bed & Breakfast (desayuno),**HB**: Half Board (media pensión: desayuno y otra comida),**FB**: Full Board (pensión completa: desayuno, almuerzo y cena)]| Categórica **ordinal** |
  | `country` | País de origen del cliente en formato ISO de 3 letras (ej., PRT, GBR, ESP). | Categórica **nominal** |
  | `market_segment` | Segmento de mercado del que procede la reserva (ej., Online TA, Offline TA/TO, Direct, Corporate, Groups, Complementary, Aviation). | Categórica **nominal** |
  | `distribution_channel` | Canal de reserva o distribución utilizado (ej., TA/TO [Travel Agents/Tour Operators], Direct, Corporate, GDS). | Categórica **nominal** |
  | `is_repeated_guest` | Huésped repetitivo (1 = sí, 0 = no) | Categórica **binaria** |
  | `previous_cancellations` | Reservas previas canceladas | Numérica **discreta** (conteo) |
  | `previous_bookings_not_canceled` | Reservas previas no canceladas | Numérica **discreta** (conteo) |
  | `reserved_room_type` | Tipo de habitación reservada (ej., A, B, C, D, etc.).| Categórica **nominal** |
  | `assigned_room_type` | Tipo de habitación asignada (puede diferir de la reservada por ascensos de categoría o sobreventa)| Categórica **nominal** |
  | `booking_changes` | Cambios realizados en la reserva | Numérica **discreta** (conteo) |
  | `deposit_type` | Garantía económica asociada a la reserva (No Deposit, Non Refund [no reembolsable], Refundable).| Categórica **nominal** |
  | `agent` | ID de la agencia de viajes que tramitó la reserva (Se almacena numéricamente pero su valor es un identificador nomimal). | Texto / identificador |
  | `company` | ID de la empresa/corporación que realizó o pagó la reserva. (Identificador nominal). | Texto / identificador |
  | `days_in_waiting_list` | Días que la reserva estuvo retenida en lista de espera antes de ser confirmada. | Numérica **discreta** |
  | `customer_type` | Tipología de la estancia y del cliente [**Transient**: Reserva individual estándar no asociada a grupo o contrato, **Contract**: Asociada a un contrato previo o tarifa negociada, **Transient-Party**: Reserva individual vinculada a otra reserva dentro de un grupo implícito, **Group**: Reserva formal de grupo]| Categórica **nominal** |
  | `adr` | Tarifa media diaria (ADR: Average Daily Rate) | Numérica **continua** |
  | `required_car_parking_spaces` | Número de cocheras/estacionamientos solicitados por el cliente. | Numérica **discreta** (conteo) |
  | `total_of_special_requests` | Peticiones especiales | Numérica **discreta** (conteo) |
  | `reservation_status` | Estado final de la reserva (Check-Out, Canceled, No-Show).| Categórica **nominal** |
  | `reservation_status_date` | Fecha del estado final | Fecha / Tiempo |
  | `booking_date` | Fecha de la reserva | Fecha / Tiempo |

### Variable target

Nuestra variable objetivo es `is_canceled`, donde:
* `1` = La reserva fue cancelada.
* `0` = La reserva no fue cancelada (se concretó o está activa).
El objetivo de negocio es predecir si una reserva será cancelada para poder tomar medidas preventivas.

## 1.3. Estructura y tipos según pandas

In [ ]:
print(f"Dimensiones del dataset de entrenamiento: {df_train.shape}")   # (registros, variables)
df_train.info(memory_usage="deep")
resumen = pd.DataFrame({
    "tipo": df_train.dtypes,
    "n_unicos": df_train.nunique(),
    "n_faltantes": df_train.isna().sum(),
    "%_faltantes": (df_train.isna().mean() * 100).round(2),
  })
resumen

## 1.4. Evaluar missing y calidad
Analizamos las variables con valores nulos (faltantes) y chequeamos posibles inconsistencias (ej. reservas sin huéspedes).

In [ ]:
# Variables con faltantes
nulos = pd.DataFrame({
    "n_faltantes": df_train.isna().sum(),
    "%_faltantes": (df_train.isna().mean() * 100).round(2).sort_values(ascending=False),
  })
print("Variables con faltantes:")
print(nulos[nulos['%_faltantes'] > 0])

# Inconsistencias: reservas sin huéspedes
sin_huespedes = df_train[(df_train['adults'] == 0) & (df_train['children'] == 0) & (df_train['babies'] == 0)]
print(f"\nReservas sin huéspedes: {len(sin_huespedes)}")

# Inconsistencias: ADR negativo
adr_negativo = df_train[df_train['adr'] < 0]
print(f"Reservas con ADR negativo: {len(adr_negativo)}")

# 2. Análisis univariado
## 2.1. Resumen estadístico de las variables numéricas y categóricas.

1.` display(df_train.describe())` Por defecto, el método .describe() procesa únicamente las columnas numéricas (int64, float64).Genera una tabla con 8 métricas estadísticas descriptivas para cada columna:

- `count`: Cantidad de valores no nulos. Útil para identificar qué variables numéricas tienen datos faltantes (por ejemplo, agent o company).
- `mean`: Promedio o media aritmética de la columna.
- `std`: Desviación estándar. Mide la dispersión o variabilidad de los datos respecto a la media.
- min: Valor mínimo registrado. Permite detectar anomalías o errores de carga (por ejemplo, valores negativos en la tarifa diaria adr).
- `25%` (Primer cuartil $Q_1$): El 25% de los datos es menor o igual a este valor.
- `50%` (Mediana o Segundo cuartil $Q_2$): El valor central que divide la distribución al 50%.
- `75%` (Tercer cuartil $Q_3$): El 75% de los datos es menor o igual a este valor.
- `max`: Valor máximo registrado. Crucial para detectar atípicos o extremos (por ejemplo, un adr de $5400$ o adults de $55$).

2. `display(df_train.describe(include=['object', 'category']))` Al pasarle el argumento include=['object', 'category'], obligamos a .describe() a evaluar las variables cualitativas o categóricas (cadenas de texto o categorías explícitas de pandas).Genera un resumen específico para el comportamiento cualitativo:

- `count`: Cantidad de registros no nulos. Permite detectar faltantes en variables de texto como country.
- `unique`: Número de categorías o valores distintos (cardinalidad de la variable). Útil para decidir transformaciones (por ejemplo, hotel tiene 2 categorías únicas, mientras que country tiene 169).
- `top`: La categoría que aparece con mayor frecuencia (la moda).
- `freq`: La frecuencia absoluta con la que aparece la categoría top.

In [ ]:
# Resumen numérico
display(df_train.describe())

# Resumen categórico
display(df_train.describe(include=['object', 'category']))

## 2.2. Análisis univariado de la variable target: `is_canceled`
Veamos la distribución de nuestra variable objetivo para entender si hay desbalance de clases.

In [ ]:
print("=== ANÁLISIS UNIVARIADO DE LA VARIABLE TARGET (is_canceled) ===\n")

# 1. Asegurar formato datetime en la fecha de reserva
df_temp = df_train.copy()
df_temp['booking_date'] = pd.to_datetime(df_temp['booking_date'])

# 2. Conteo y Proporciones
counts = df_temp['is_canceled'].value_counts()
percentages = df_temp['is_canceled'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    'Cantidad': counts,
    'Porcentaje (%)': percentages.round(2)
})
summary_df.index = summary_df.index.map({0: 'No Cancelada (0)', 1: 'Cancelada (1)'})
print(summary_df)
print("\n-----------------------------------------------------------")

# 3. Ratio de Desbalance
imbalance_ratio = counts[0] / counts[1]
print(f"Ratio de Desbalance (No Canceladas / Canceladas): {imbalance_ratio:.2f} : 1")

# 4. Visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Barras con conteo y % (Sin warnings en Seaborn/Matplotlib)
palette = ['#2b5c8f', '#d95f02']
sns.barplot(
    x=counts.index, 
    y=counts.values, 
    hue=counts.index, 
    palette=palette, 
    legend=False, 
    ax=axes[0]
)
axes[0].set_title('Distribución de Frecuencia del Target', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Estado de Reserva', fontsize=10)
axes[0].set_ylabel('Número de Reservas', fontsize=10)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['No Cancelada (0)', 'Cancelada (1)'])

for p in axes[0].patches:
    h = p.get_height()
    axes[0].annotate(
        f'{int(h):,}\n({h/len(df_temp):.1%})',
        (p.get_x() + p.get_width() / 2., h / 2),
        ha='center', va='center', color='white', fontweight='bold'
    )
                      
# Gráfico 2: Evolución temporal por booking_date (Corrección del accesor .dt)
df_temp['year_month'] = df_temp['booking_date'].dt.to_period('M')
rate_monthly = df_temp.groupby('year_month')['is_canceled'].mean() * 100

rate_monthly.plot(kind='line', ax=axes[1], marker='o', color='#d95f02', linewidth=2)
axes[1].set_title('Evolución Mensual de la Tasa de Cancelación (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fecha de Reserva (Mes/Año)', fontsize=10)
axes[1].set_ylabel('Tasa de Cancelación (%)', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Ratio de Desbalance (Imbalance Ratio)

Es una métrica simple que expresa cuántas instancias de la clase mayoritaria (en este caso, reservas no canceladas $0$) existen por cada instancia de la clase minoritaria (reservas canceladas $1$).

$$\text{Ratio de Desbalance} = \frac{\text{Cantidad de clase 0 (No Canceladas)}}{\text{Cantidad de clase 1 (Canceladas)}} = \frac{53,830}{20,456} \approx 2.63$$

#### ¿Cómo interpretar este $2.63 : 1$ en la práctica?
1. Interpretación directa: Significa que por cada reserva que se cancela en el dataset de entrenamiento, el hotel registra aproximadamente 2.63 reservas efectivas.
2. Severidad del desbalance:
  - Balanceado (1:1 a 1.5:1): Las clases están representadas casi por igual.
  - Desbalance Moderado (2:1 a 9:1): Aquí se ubica nuestro proyecto (2.63:1). Hay una clase predominante, pero la clase minoritaria sigue teniendo suficiente representación matemática ($27.5\%$).
  - Desbalance Severo (10:1 a 1000:1 o más): Típico en detección de fraudes con tarjetas de crédito o diagnóstico de enfermedades raras ($0.1\%$ de positivos).

#### Implicaciones para las Fases de Scikit-Learn
Haber calculado este ratio de $2.63 : 1$ nos dicta tres decisiones clave para la construcción de nuestros pipelines y modelos:
1. No necesitamos Over-sampling / Under-sampling agresivo: No hace falta usar técnicas complejas como SMOTE o Random Under Sampling, ya que contamos con más de 20,000 ejemplos de la clase de interés ($1$).
2. Ajuste de pesos en los algoritmos (class_weight='balanced'): En modelos como RandomForestClassifier o LogisticRegression, pasaremos el parámetro class_weight='balanced'. Esto le dice al algoritmo que le dé un peso ligeramente mayor (aproximadamente $2.63$ veces más) a las penalizaciones cuando se equivoca prediciendo una cancelación.
3. Selección de Métricas de Evaluación: Si un modelo predijera siempre "No Cancelado" (cero inteligente), obtendría un $72.5\%$ de Accuracy (exactitud), lo cual sería una métrica engañosa. Por ello, evaluaremos con ROC-AUC, Precision, Recall y el F1-Score de la clase $1$.

### Evolución Mensual de la Tasa de Cancelación (%) según la fecha de reserva (`booking_date`)

1. Distorsión inicial por volumen de muestra (Años 2013-2014)
- Comportamiento: Se observan picos pronunciados y volatilidad extrema entre 2013 y finales de 2014 (alcanzando un pico superior al 80% en octubre de 2014).
- Causa de Negocio: Durante este período hay un número insignificante de reservas generadas (apenas entre 1 y 193 reservas por mes, comparado con las 3,000–6,000 mensuales de 2016 y 2017).
- Impacto en ML: Es un sesgo común en datos históricos antiguos. En la etapa de preprocesamiento evaluaremos si filtrar o penalizar estas pocas filas ruidosas pre-2015 para evitar distorsionar los patrones de los modelos.

2. Estabilización de la tasa en el rango 25% – 33% (Periodo Estable: 2016–2017)
- Comportamiento: A partir de enero de 2016, la tasa de cancelación mensual de las reservas se estabiliza de manera notable dentro de una franja de entre el 26% y el 33%.
- Diagnóstico: Indica que el negocio opera bajo un comportamiento estocástico predecible y que no hubo cambios drásticos en las políticas del hotel ni eventos macroeconómicos disruptivos en esos años.

3. Ausencia de Data Leakage en la División Temporal
- Comportamiento: En el tramo final del conjunto de entrenamiento (principios de 2017), la tasa se sitúa en torno al 27.6% - 29.8%, un valor casi idéntico al promedio general de entrenamiento (27.5%) y al del conjunto de test (27.2%).
- Diagnóstico: Confirma que la partición temporal que realizamos es sólida: el modelo no sufrirá concept drift (cambio de concepto o distribución) severo cuando pase a evaluar el conjunto de prueba (test.csv).

## 2.3. Análisis univariado de variables numéricas predictoras

1. `lead_time` (Días de anticipación de la reserva)
- Forma de la distribución: Presenta una asimetría positiva severa (right-skewed). La mayoría de las reservas se realizan con poca anticipación (mediana = 56 días), pero existe una "cola larga" que se extiende hasta 737 días (más de 2 años de anticipación).
- Impacto en ML: Modelos lineales (como Regresión Logística) se benefician de transformaciones logarítmicas o escalados robustos (RobustScaler) en esta variable para atenuar la asimetría.

2. `adr` (Average Daily Rate / Tarifa diaria promedio)Comportamiento general: 
- Presenta una distribución aproximadamente unimodal concentrada entre 68.5 y 126 EUR (mediana = 93.6 EUR).
- Detección de Atípicos (Outliers): 
  - Mínimos anómalos: Existen valores negativos (mínimo = -6.38 EUR) y valores en $0.00$ EUR (reservas complementarias o errores).
  - Máximo atípico severo: Registra un valor extremo de 5,400 EUR (mientras el percentil 99 está en 242 EUR).
- Decisión de Preparación de Datos: En la etapa de limpieza se deberá aplicar un filtro de límites lógicos (por ejemplo, reemplazar o eliminar tarifas $< 0$ o $> 1000$ EUR) o aplicar un escalador insensible a atípicos.

3. Noches de Estadía (`stays_in_week_nights` y `stays_in_weekend_nights`)
- Comportamiento: La aplastante mayoría de los huéspedes se aloga entre 1 y 4 noches totales.
- Atípicos: Existen valores extremos como 50 noches de semana y 19 noches de fin de semana.
- Ingeniería de Características (Feature Engineering): Conviene crear la variable consolidada total_nights = stays_in_week_nights + stays_in_weekend_nights.

4. Ocupantes (`adults`, `children`, `babies`)
- adults: La mediana es de 2 adultos (modalidad de pareja). Sin embargo, hay registros anómalos con hasta 55 adultos en una sola reserva o reservas con 0 adultos.
- children y babies: Distribuciones altamente dispersas compuestas principalmente por ceros. children contiene 4 valores nulos en el dataset original que imputaremos con 0.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Lead Time
sns.histplot(df_train['lead_time'], kde=True, ax=axes[0, 0], color='#2b5c8f', bins=40,)
axes[0, 0].set_title('Distribución de Lead Time (Días de Anticipación)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Días')
axes[0, 0].set_ylabel('Frecuencia')

# 2. ADR (Filtrando atípicos extremos para visualización)
sns.histplot(df_train[df_train['adr'] < 400]['adr'], kde=True, ax=axes[0, 1], color='#d95f02', bins=40)
axes[0, 1].set_title('Distribución de ADR (Tarifa Diaria Promedio < 400)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Precio (EUR)')
axes[0, 1].set_ylabel('Frecuencia')

# 3. Noches de estadía total
total_nights = df_train['stays_in_week_nights'] + df_train['stays_in_weekend_nights']
sns.histplot(total_nights[total_nights <= 14], discrete=True, ax=axes[1, 0], color='#2ca02c')
axes[1, 0].set_title('Distribución de Noches Totales (<= 14 noches)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Noches')
axes[1, 0].set_ylabel('Frecuencia')

# 4. Adultos
sns.countplot(x='adults', data=df_train[df_train['adults'] <= 5], ax=axes[1, 1], color='#82589B')
axes[1, 1].set_title('Distribución de Cantidad de Adultos (<= 5)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Número de Adultos')
axes[1, 1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()
